# 知识图谱工程：从 ontology、抽取到可引用的 KG-RAG

知识图谱（KG）不是把若干 `(subject, predicate, object)` 写入图数据库就结束。可用系统还要回答：实体 ID 如何稳定、同名如何消歧、事实来自哪份文档、何时有效、属于哪个 tenant、重复摄取是否幂等、删除是否可审计、查询路径能否回到原始证据。

本 Notebook 用规则抽取 + NetworkX `MultiDiGraph` 实现一个小型供应链图谱，覆盖实体规范化、关系约束、provenance、confidence、时间、tenant、upsert/delete、1-hop/多跳/路径、证据子图与 KG-RAG context。所有数据均为虚构教学数据；规则抽取和内存图不能替代生产 NER/RE、实体解析服务、持久图数据库或权限系统。

## 学习目标

1. 从业务问题反推 schema/ontology，而不是先画一张巨大的图；
2. 区分实体、关系、属性和事件，并为事实保留来源与有效时间；
3. 实现可拒绝歧义的实体规范化、关系约束和幂等写入；
4. 执行 1-hop、多跳和路径查询，构造带引用的证据子图；
5. 评估抽取、链接、查询和证据质量，并理解生产图数据库取舍。


## 1. 从业务问题设计 schema / ontology

本例服务两个可验证问题：

- “谁供应 R100 路由器？”——需要 `Organization -SUPPLIES-> Product`；
- “INC-7 影响哪些客户，证据是什么？”——需要 `Incident -AFFECTS-> Product <-BUYS- Organization`，并能沿路径回到来源文档。

因此最小 ontology 包含 `Organization、Product、Contract、Incident` 四类实体，以及 `SUPPLIES、BUYS、PARTY_TO、COVERS、AFFECTS` 五类有向关系。每条关系定义允许的起止类型。不要因为文本出现了一个名词就新增类型；schema 变更应有版本、迁移、兼容与负责人。

开放世界假设下，“图中没有”通常等于“目前没有证据”，不等于事实为假。强制唯一性、基数和必填属性时，要明确它是业务规则还是世界知识，避免把数据缺失误判成现实否定。


In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import dataclass, field, replace
from datetime import datetime, timezone
import hashlib
import re
import unicodedata
from typing import Sequence

import networkx as nx
import pandas as pd

SCHEMA_VERSION = "supply-kg-v1"
ENTITY_TYPES = {"Organization", "Product", "Contract", "Incident"}
RELATION_SIGNATURES = {
    "SUPPLIES": ("Organization", "Product"),
    "BUYS": ("Organization", "Product"),
    "PARTY_TO": ("Organization", "Contract"),
    "COVERS": ("Contract", "Product"),
    "AFFECTS": ("Incident", "Product"),
}

@dataclass(frozen=True)
class EntityRecord:
    entity_id: str
    entity_type: str
    canonical_name: str
    tenant: str
    aliases: tuple[str, ...] = ()
    properties: dict = field(default_factory=dict)

@dataclass(frozen=True)
class FactRecord:
    fact_id: str
    subject_id: str
    predicate: str
    object_id: str
    tenant: str
    source_id: str
    source_uri: str
    source_span: tuple[int, int]
    extractor_version: str
    confidence: float
    observed_at: str
    valid_from: str | None = None
    valid_to: str | None = None

@dataclass(frozen=True)
class AuthContext:
    """由认证网关构造的可信上下文；tenant/scopes 不能取自请求正文。"""
    tenant: str
    scopes: frozenset[str]
    principal_id: str = "demo-principal"

    def require(self, *required_scopes: str) -> None:
        missing = set(required_scopes) - set(self.scopes)
        if missing:
            raise PermissionError(f"缺少 scope: {sorted(missing)}")

def parse_iso_time(value: str, field_name: str = "time") -> datetime:
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{field_name} 必须是非空 ISO-8601 字符串")
    try:
        parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError as exc:
        raise ValueError(f"{field_name} 不是合法 ISO-8601 时间: {value}") from exc
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)
    return parsed.astimezone(timezone.utc)

def stable_id(*parts: str) -> str:
    payload = "|".join(str(part) for part in parts)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:20]

print("schema=", SCHEMA_VERSION, "entity types=", sorted(ENTITY_TYPES))


## 2. 实体、关系、属性与事件怎么分

- **实体**有跨文档稳定身份，例如组织、产品、合同；名称只是属性，不应拿可变名称当主键。
- **关系**连接两个实体，并且本身常有来源、置信度和有效时间。若这些限定信息重要，关系不能只存一个字符串。
- **属性**描述单个实体，例如产品型号、事件严重度。高频变化或需要独立溯源的“属性”可能更适合建成实体/关系。
- **事件**不是普通标签。事件有 ID、参与者、发生时间、地点和状态，通常建成一等实体，才能连接多方并保留事件生命周期。

本例把 `Incident` 作为事件实体；`AFFECTS` 连接受影响产品。事实的 `observed_at` 表示系统何时观察到证据，`valid_from/valid_to` 表示事实声称何时有效，两者不能混用。


In [ ]:
BASE_ENTITIES = [
    EntityRecord("org:huaxing", "Organization", "华星科技有限公司", "tenant-a", ("华星科技", "华星科技（北京）有限公司", "华星")),
    EntityRecord("org:huaxing-lab", "Organization", "华星研究院", "tenant-a", ("华星",)),
    EntityRecord("org:farsea", "Organization", "远海银行", "tenant-a", ("远海",)),
    EntityRecord("product:r100", "Product", "R100路由器", "tenant-a", ("R100", "R-100路由器"), {"model": "R100"}),
    EntityRecord("contract:c202601", "Contract", "C-2026-01", "tenant-a", (), {"contract_value": 800000, "currency": "CNY"}),
    EntityRecord("incident:inc7", "Incident", "INC-7", "tenant-a", ("固件漏洞INC-7",), {"severity": "high", "occurred_at": "2026-07-01"}),
    # 同名组织在另一个 tenant 中拥有不同 ID，绝不能跨租户合并。
    EntityRecord("org:huaxing-b", "Organization", "华星科技有限公司", "tenant-b", ("华星科技",)),
    EntityRecord("product:r100-b", "Product", "R100路由器", "tenant-b", ("R100",)),
]

def normalize_alias(name: str) -> str:
    name = unicodedata.normalize("NFKC", name).lower().replace("—", "-")
    return re.sub(r"[\s()（）·]", "", name)

alias_index = defaultdict(list)
entity_lookup = {}
for entity in BASE_ENTITIES:
    entity_lookup[(entity.tenant, entity.entity_id)] = entity
    for alias in (entity.canonical_name, *entity.aliases):
        alias_index[(entity.tenant, normalize_alias(alias))].append(entity.entity_id)

def resolve_mention(mention: str, tenant: str, expected_type: str | None = None) -> str | None:
    candidates = alias_index.get((tenant, normalize_alias(mention)), [])
    if expected_type:
        candidates = [cid for cid in candidates if entity_lookup[(tenant, cid)].entity_type == expected_type]
    # 歧义时返回 None，不用“第一个候选”悄悄污染图谱。
    return candidates[0] if len(candidates) == 1 else None

assert resolve_mention("华星科技（北京）有限公司", "tenant-a", "Organization") == "org:huaxing"
assert resolve_mention("华星", "tenant-a", "Organization") is None
assert resolve_mention("华星科技", "tenant-b", "Organization") == "org:huaxing-b"
print("规范化示例通过；歧义简称‘华星’被安全拒绝。")


## 3. 规则式抽取：高精度起点，不是假装通用 IE

规则适合格式稳定的合同字段、CMDB 或业务表单：可解释、易回归、置信度较高。自由文本中的指代、省略、否定和跨句关系则需要 NER/RE、LLM 或人工复核。无论哪种抽取器，都应输出 mention span、抽取器版本和置信度，而不是只输出三元组。

实体解析通常经过 normalization -> 候选生成 -> 类型/上下文排序 -> 阈值/拒绝 -> 人工反馈。去重不能只比较名称；统一社会信用代码、产品型号、地址和上下文往往比编辑距离更可靠。别名表必须 tenant-scoped，避免把两个客户空间的同名对象合并。


In [ ]:
SOURCE_DOCUMENTS = [
    {"source_id": "d1", "tenant": "tenant-a", "uri": "contracts/supply-note-1", "observed_at": "2026-01-02", "valid_from": "2026-01-01", "text": "供应商：华星科技有限公司；产品：R100路由器；关系：供应。"},
    {"source_id": "d2", "tenant": "tenant-a", "uri": "orders/order-88", "observed_at": "2026-02-01", "valid_from": "2026-02-01", "text": "客户：远海银行；产品：R100；关系：采购。"},
    {"source_id": "d3", "tenant": "tenant-a", "uri": "contracts/C-2026-01", "observed_at": "2026-01-03", "valid_from": "2026-01-01", "text": "合同：C-2026-01；甲方：远海银行；乙方：华星科技有限公司；产品：R100路由器；生效：2026-01-01。"},
    {"source_id": "d4", "tenant": "tenant-a", "uri": "incidents/INC-7", "observed_at": "2026-07-02", "valid_from": "2026-07-01", "text": "事件：INC-7；产品：R-100路由器；类型：固件漏洞；发生：2026-07-01。"},
    # d5 是同一供应事实的第二份证据，用来验证 upsert 聚合 provenance。
    {"source_id": "d5", "tenant": "tenant-a", "uri": "catalog/R100", "observed_at": "2026-03-01", "valid_from": "2026-01-01", "text": "供应商：华星科技（北京）有限公司；产品：R100；关系：供应。"},
    {"source_id": "d6", "tenant": "tenant-b", "uri": "catalog/private-R100", "observed_at": "2026-03-02", "valid_from": "2026-03-01", "text": "供应商：华星科技；产品：R100；关系：供应。"},
]

def parse_fields(text: str) -> dict[str, str]:
    return {key.strip(): value.strip() for key, value in re.findall(r"([^：；]+)：([^；。]+)", text)}

def make_fact(doc: dict, subject_id: str, predicate: str, object_id: str, confidence: float) -> FactRecord:
    fact_id = stable_id(doc["tenant"], subject_id, predicate, object_id, doc.get("valid_from") or "")
    return FactRecord(
        fact_id, subject_id, predicate, object_id, doc["tenant"],
        doc["source_id"], doc["uri"], (0, len(doc["text"])), "rules-v1", confidence,
        doc["observed_at"], doc.get("valid_from"), doc.get("valid_to"),
    )

def extract_facts(doc: dict) -> list[FactRecord]:
    fields, tenant = parse_fields(doc["text"]), doc["tenant"]
    facts = []
    product = resolve_mention(fields.get("产品", ""), tenant, "Product")
    if fields.get("关系") == "供应":
        supplier = resolve_mention(fields.get("供应商", ""), tenant, "Organization")
        if supplier and product:
            facts.append(make_fact(doc, supplier, "SUPPLIES", product, 0.99))
    if fields.get("关系") == "采购":
        customer = resolve_mention(fields.get("客户", ""), tenant, "Organization")
        if customer and product:
            facts.append(make_fact(doc, customer, "BUYS", product, 0.99))
    if "合同" in fields:
        contract = resolve_mention(fields["合同"], tenant, "Contract")
        for party_field in ("甲方", "乙方"):
            party = resolve_mention(fields.get(party_field, ""), tenant, "Organization")
            if party and contract:
                facts.append(make_fact(doc, party, "PARTY_TO", contract, 0.98))
        if contract and product:
            facts.append(make_fact(doc, contract, "COVERS", product, 0.97))
    if "事件" in fields:
        incident = resolve_mention(fields["事件"], tenant, "Incident")
        if incident and product:
            facts.append(make_fact(doc, incident, "AFFECTS", product, 0.96))
    return facts

extracted_facts = [fact for doc in SOURCE_DOCUMENTS for fact in extract_facts(doc)]
display(pd.DataFrame([fact.__dict__ for fact in extracted_facts])[["fact_id", "subject_id", "predicate", "object_id", "tenant", "source_id", "confidence"]])


## 4. provenance、confidence、time、tenant 与约束校验

三元组的最小生产记录应包含：规范化两端、关系类型、tenant、source_id/URI、原文 span、抽取器版本、confidence、observed_at 与有效时间。confidence 表示抽取/链接不确定性，不是真实世界发生概率；多份同源拷贝也不能简单当作独立证据相乘。

写入前执行类似 SHACL 的约束：实体存在、同 tenant、关系签名正确、置信度在 `[0,1]`、来源和时间格式完整。严重约束失败进入 quarantine，不应为了“图更大”静默修复。跨 tenant 边即使类型正确也必须拒绝。


In [ ]:
def validate_entity(entity: EntityRecord) -> None:
    if entity.entity_type not in ENTITY_TYPES:
        raise ValueError(f"未知实体类型: {entity.entity_type}")
    if not entity.entity_id or not entity.canonical_name or not entity.tenant:
        raise ValueError("实体 ID、名称和 tenant 必填")

def validate_fact(fact: FactRecord, entities: dict[tuple[str, str], EntityRecord]) -> None:
    if fact.predicate not in RELATION_SIGNATURES:
        raise ValueError(f"未知关系: {fact.predicate}")
    subject = entities.get((fact.tenant, fact.subject_id))
    object_ = entities.get((fact.tenant, fact.object_id))
    if subject is None or object_ is None:
        raise ValueError("关系两端必须存在于同一 tenant")
    expected = RELATION_SIGNATURES[fact.predicate]
    actual = (subject.entity_type, object_.entity_type)
    if actual != expected:
        raise ValueError(f"关系签名错误: {fact.predicate} 期望 {expected}，实际 {actual}")
    if not 0.0 <= fact.confidence <= 1.0 or not fact.source_id or not fact.source_uri:
        raise ValueError("confidence 必须在 [0,1] 且 provenance 必填")
    if len(fact.source_span) != 2 or fact.source_span[0] < 0 or fact.source_span[1] < fact.source_span[0]:
        raise ValueError("source_span 必须是非负且有序的 (start, end)")
    parse_iso_time(fact.observed_at, "observed_at")
    valid_from = parse_iso_time(fact.valid_from, "valid_from") if fact.valid_from else None
    valid_to = parse_iso_time(fact.valid_to, "valid_to") if fact.valid_to else None
    if valid_from and valid_to and valid_from > valid_to:
        raise ValueError("valid_from 不能晚于 valid_to")

for entity in BASE_ENTITIES:
    validate_entity(entity)
for fact in extracted_facts:
    validate_fact(fact, entity_lookup)

invalid_error = None
try:
    validate_fact(replace(extracted_facts[0], predicate="COVERS"), entity_lookup)
except ValueError as exc:
    invalid_error = str(exc)
print("非法 Organization-COVERS-Product 被拒绝：", invalid_error)


## 5. NetworkX 存储与幂等 upsert/delete

节点内部 key 使用 `tenant|entity_id`，边 key 使用事实自然键的稳定 hash。同一事实重复摄取不会增加边，而是按 source_id 合并来源；较新的 observed_at 可以更正同一自然事实的 valid_to，并留下字段级审计记录。`MultiDiGraph` 允许同一实体对存在不同关系或不同时段的事实。

删除默认做逻辑 tombstone 并写审计记录，而不是物理抹掉证据。重新出现同一自然键时是否复活必须有显式策略；本教学实现不会自动复活 tombstone。生产系统还需要事务、唯一约束、并发控制、重放 checkpoint 和 dead-letter queue。


In [ ]:
class KnowledgeGraphStore:
    def __init__(self):
        self.graph = nx.MultiDiGraph(schema_version=SCHEMA_VERSION)
        self.entities: dict[tuple[str, str], EntityRecord] = {}
        self.audit_log: list[dict] = []

    @staticmethod
    def node_key(tenant: str, entity_id: str) -> str:
        return f"{tenant}|{entity_id}"

    def upsert_entity(self, entity: EntityRecord) -> None:
        validate_entity(entity)
        lookup_key = (entity.tenant, entity.entity_id)
        previous = self.entities.get(lookup_key)
        if previous and previous.entity_type != entity.entity_type:
            raise ValueError("稳定 entity_id 不允许改变类型")
        self.entities[lookup_key] = entity
        self.graph.add_node(
            self.node_key(entity.tenant, entity.entity_id),
            entity_id=entity.entity_id, entity_type=entity.entity_type,
            canonical_name=entity.canonical_name, tenant=entity.tenant,
            aliases=entity.aliases, properties=dict(entity.properties),
        )

    def upsert_fact(self, fact: FactRecord) -> None:
        validate_fact(fact, self.entities)
        subject = self.node_key(fact.tenant, fact.subject_id)
        object_ = self.node_key(fact.tenant, fact.object_id)
        provenance = {
            "source_id": fact.source_id, "source_uri": fact.source_uri,
            "source_span": fact.source_span, "extractor_version": fact.extractor_version,
            "observed_at": fact.observed_at, "confidence": fact.confidence,
            "valid_from": fact.valid_from, "valid_to": fact.valid_to,
        }
        existing = self.graph.get_edge_data(subject, object_, key=fact.fact_id)
        changes = {}
        if existing is not None:
            existing["provenance"][fact.source_id] = provenance
            existing["confidence"] = max(item["confidence"] for item in existing["provenance"].values())
            if parse_iso_time(fact.observed_at, "observed_at") > parse_iso_time(existing["last_observed_at"], "last_observed_at"):
                if fact.valid_to != existing.get("valid_to"):
                    changes["valid_to"] = {"old": existing.get("valid_to"), "new": fact.valid_to}
                    existing["valid_to"] = fact.valid_to
                existing["last_observed_at"] = fact.observed_at
            operation = "update_temporal" if changes else "merge_provenance"
        else:
            self.graph.add_edge(
                subject, object_, key=fact.fact_id, fact_id=fact.fact_id,
                predicate=fact.predicate, tenant=fact.tenant, confidence=fact.confidence,
                valid_from=fact.valid_from, valid_to=fact.valid_to,
                last_observed_at=fact.observed_at, status="active",
                provenance={fact.source_id: provenance},
            )
            operation = "insert"
        audit_entry = {
            "operation": operation, "tenant": fact.tenant, "fact_id": fact.fact_id,
            "source_id": fact.source_id, "observed_at": fact.observed_at,
        }
        if changes:
            audit_entry["changes"] = changes
        self.audit_log.append(audit_entry)

    def active_facts(self, tenant: str, as_of: str | None = None):
        rows = []
        for subject, object_, key, data in self.graph.edges(keys=True, data=True):
            if data["tenant"] != tenant or data["status"] != "active":
                continue
            if as_of:
                point = parse_iso_time(as_of, "as_of")
                valid_from = parse_iso_time(data["valid_from"], "valid_from") if data.get("valid_from") else None
                valid_to = parse_iso_time(data["valid_to"], "valid_to") if data.get("valid_to") else None
                if (valid_from and point < valid_from) or (valid_to and point > valid_to):
                    continue
            rows.append((subject, object_, key, data))
        return rows

    def delete_fact(self, tenant: str, fact_id: str, deleted_at: str) -> bool:
        for subject, object_, key, data in self.graph.edges(keys=True, data=True):
            if data["tenant"] == tenant and key == fact_id:
                if data["status"] == "tombstoned":
                    return False
                data["status"], data["deleted_at"] = "tombstoned", deleted_at
                self.audit_log.append({"operation": "tombstone", "fact_id": fact_id, "deleted_at": deleted_at})
                return True
        return False

store = KnowledgeGraphStore()
for entity in BASE_ENTITIES:
    store.upsert_entity(entity)
for fact in extracted_facts:
    store.upsert_fact(fact)

supply_fact_id = next(f.fact_id for f in extracted_facts if f.tenant == "tenant-a" and f.predicate == "SUPPLIES")
supply_edge = next(data for _, _, key, data in store.graph.edges(keys=True, data=True) if key == supply_fact_id)
initial_supply_provenance_count = len(supply_edge["provenance"])
print({"nodes": store.graph.number_of_nodes(), "edges": store.graph.number_of_edges(), "supply_provenance_count": initial_supply_provenance_count})


## 6. 1-hop、多跳与路径查询

1-hop 查询适合直接邻接事实，例如产品的供应商。多跳查询必须把每一步写成 `(predicate, direction)` 模式，并在遍历前限制深度、时间点和 AuthContext tenant；路径结果还要携带精确 fact_id，不能先投影成无向简单图再猜回原边。

路径只是候选解释，不自动等于因果关系。`客户 -BUYS-> 产品 <-AFFECTS- 事件` 支持“客户购买的产品受事件影响”，但不能推出客户已经发生损失。生产查询应使用预定义模板或受约束的图查询，而不是让 LLM 任意生成并执行 Cypher/SPARQL。


In [ ]:
AUTH_A_READER = AuthContext("tenant-a", frozenset({"kg.read"}), "reader-a")
AUTH_A_EVIDENCE = AuthContext("tenant-a", frozenset({"kg.read", "kg.provenance.read"}), "rag-a")

def _entity_from_node(store: KnowledgeGraphStore, node_key: str) -> EntityRecord:
    data = store.graph.nodes[node_key]
    return store.entities[(data["tenant"], data["entity_id"])]

def project_entity(entity: EntityRecord, auth: AuthContext) -> dict:
    if entity.tenant != auth.tenant:
        raise PermissionError("实体 tenant 与认证上下文不一致")
    properties = dict(entity.properties)
    if "contract.finance.read" not in auth.scopes:
        properties.pop("contract_value", None)
        properties.pop("currency", None)
    return {
        "entity_id": entity.entity_id, "name": entity.canonical_name,
        "type": entity.entity_type, "tenant": entity.tenant, "properties": properties,
    }

def secure_one_hop(
    store: KnowledgeGraphStore, auth: AuthContext, entity_id: str,
    predicate: str, direction: str, as_of: str | None = None,
) -> list[dict]:
    auth.require("kg.read")
    if predicate not in RELATION_SIGNATURES or direction not in {"out", "in"}:
        raise ValueError("predicate 未注册或 direction 不是 out/in")
    center = store.node_key(auth.tenant, entity_id)
    neighbor_keys = set()
    for subject, object_, _, data in store.active_facts(auth.tenant, as_of):
        if data["predicate"] != predicate:
            continue
        if direction == "out" and subject == center:
            neighbor_keys.add(object_)
        elif direction == "in" and object_ == center:
            neighbor_keys.add(subject)
    return [project_entity(_entity_from_node(store, key), auth) for key in sorted(neighbor_keys)]

@dataclass(frozen=True)
class PathStep:
    from_node: str
    to_node: str
    subject_node: str
    object_node: str
    fact_id: str
    predicate: str
    direction: str

def _matching_steps(
    store: KnowledgeGraphStore, auth: AuthContext, current_node: str,
    predicate: str, direction: str, as_of: str,
) -> list[PathStep]:
    candidates = []
    for subject, object_, fact_id, data in store.active_facts(auth.tenant, as_of):
        if data["predicate"] != predicate:
            continue
        if direction == "out" and subject == current_node:
            next_node = object_
        elif direction == "in" and object_ == current_node:
            next_node = subject
        else:
            continue
        candidates.append(PathStep(current_node, next_node, subject, object_, fact_id, predicate, direction))
    return sorted(candidates, key=lambda step: (step.fact_id, step.to_node))

def constrained_path(
    store: KnowledgeGraphStore, auth: AuthContext, source_id: str, target_id: str,
    as_of: str, allowed_steps: Sequence[tuple[str, str]], max_hops: int = 3,
) -> tuple[PathStep, ...]:
    auth.require("kg.read")
    pattern = tuple(allowed_steps)
    if not pattern or len(pattern) > max_hops:
        raise ValueError(f"路径模式长度必须在 1..{max_hops}")
    for predicate, direction in pattern:
        if predicate not in RELATION_SIGNATURES or direction not in {"out", "in"}:
            raise ValueError(f"非法路径步骤: {(predicate, direction)}")
    source = store.node_key(auth.tenant, source_id)
    target = store.node_key(auth.tenant, target_id)
    frontier: list[tuple[str, tuple[PathStep, ...]]] = [(source, tuple())]
    for predicate, direction in pattern:
        next_frontier = []
        for current, path_so_far in frontier:
            visited = {source, *(step.to_node for step in path_so_far)}
            for step in _matching_steps(store, auth, current, predicate, direction, as_of):
                if step.to_node not in visited:
                    next_frontier.append((step.to_node, path_so_far + (step,)))
        frontier = next_frontier
        if not frontier:
            break
    matches = [path for node, path in frontier if node == target]
    if not matches:
        raise nx.NetworkXNoPath("不存在满足 predicate/direction 模式的路径")
    return min(matches, key=lambda path: tuple(step.fact_id for step in path))

suppliers = secure_one_hop(store, AUTH_A_READER, "product:r100", "SUPPLIES", "in", as_of="2026-07-10")
affected_products = secure_one_hop(store, AUTH_A_READER, "incident:inc7", "AFFECTS", "out", as_of="2026-07-10")
affected_customers = []
for product in affected_products:
    affected_customers.extend(secure_one_hop(store, AUTH_A_READER, product["entity_id"], "BUYS", "in", as_of="2026-07-10"))

# 回归夹具：同一组织与产品间再放一条置信度更高的 SUPPLIES，路径仍必须精确选择 BUYS。
parallel_supplies_fact = FactRecord(
    stable_id("tenant-a", "org:farsea", "SUPPLIES", "product:r100", "2026-02-01"),
    "org:farsea", "SUPPLIES", "product:r100", "tenant-a", "parallel-supply",
    "regression/parallel-supply", (0, 18), "regression-v1", 1.0,
    "2026-02-03", "2026-02-01", None,
)
store.upsert_fact(parallel_supplies_fact)

INCIDENT_CUSTOMER_PATTERN = (("AFFECTS", "out"), ("BUYS", "in"))
evidence_path = constrained_path(
    store, AUTH_A_READER, "incident:inc7", "org:farsea", "2026-07-10",
    INCIDENT_CUSTOMER_PATTERN, max_hops=2,
)
path_nodes = [evidence_path[0].from_node, *(step.to_node for step in evidence_path)]
print("R100 供应商：", [entity["name"] for entity in suppliers])
print("INC-7 关联客户：", [entity["name"] for entity in affected_customers])
print("受约束证据路径：", [(step.predicate, step.direction, step.fact_id) for step in evidence_path])
print("路径实体：", [_entity_from_node(store, key).canonical_name for key in path_nodes])


## 7. 证据子图与 KG-RAG context

KG-RAG 不应只把实体名称列表塞进 prompt。正确的 context 至少包含：查询相关的最小子图、边方向/关系、事实有效时间、confidence、source URI 和稳定 citation ID。回答中的每个关键 claim 都应映射到一条或多条边及其原始来源。

图谱结构可以提高多跳检索的可控性，但不会自动保证事实正确：错误实体链接会把路径整体带偏，过期事实会生成过期答案，恶意来源也可能污染图。context assembly 前仍要执行 ACL、时间过滤、来源信誉、去重和预算控制。


In [ ]:
PREDICATE_ZH = {"SUPPLIES": "供应", "BUYS": "采购", "PARTY_TO": "签约方", "COVERS": "覆盖", "AFFECTS": "影响"}

def exact_facts_for_path(
    store: KnowledgeGraphStore, auth: AuthContext,
    path: Sequence[PathStep], as_of: str,
) -> list[tuple[PathStep, dict]]:
    auth.require("kg.read", "kg.provenance.read")
    active = store.active_facts(auth.tenant, as_of)
    selected = []
    for step in path:
        matches = [
            data for subject, object_, fact_id, data in active
            if fact_id == step.fact_id and subject == step.subject_node
            and object_ == step.object_node and data["predicate"] == step.predicate
        ]
        expected_from = step.subject_node if step.direction == "out" else step.object_node
        expected_to = step.object_node if step.direction == "out" else step.subject_node
        if len(matches) != 1 or step.from_node != expected_from or step.to_node != expected_to:
            raise ValueError(f"路径步骤与精确事实不一致: {step.fact_id}")
        selected.append((step, matches[0]))
    return selected

def _best_provenance(data: dict) -> dict:
    source = max(
        data["provenance"].values(),
        key=lambda item: (item["confidence"], parse_iso_time(item["observed_at"]), item["source_id"]),
    )
    if abs(source["confidence"] - data["confidence"]) > 1e-12:
        raise AssertionError("事实 confidence 必须等于可引用来源的最高 confidence")
    return source

def assemble_kg_context(
    store: KnowledgeGraphStore, auth: AuthContext,
    path: Sequence[PathStep], as_of: str,
) -> tuple[str, dict[str, dict]]:
    blocks, citation_map = [], {}
    for step, data in exact_facts_for_path(store, auth, path, as_of):
        source = _best_provenance(data)
        citation = f"{step.fact_id}/{source['source_id']}"
        subject_name = _entity_from_node(store, step.subject_node).canonical_name
        object_name = _entity_from_node(store, step.object_node).canonical_name
        blocks.append(
            f"[{citation}] {subject_name} --{PREDICATE_ZH[step.predicate]}--> {object_name}；"
            f"traversal={step.direction}；有效期={data.get('valid_from')}..{data.get('valid_to') or 'open'}；"
            f"confidence={source['confidence']:.2f}；source={source['source_uri']}"
        )
        citation_map[citation] = {
            "fact_id": step.fact_id, "predicate": step.predicate,
            "direction": step.direction, **source,
        }
    return "\n".join(blocks), citation_map

kg_context, citation_map = assemble_kg_context(store, AUTH_A_EVIDENCE, evidence_path, "2026-07-10")
answer_citations = list(citation_map)
answer = (
    f"远海银行采购了受 INC-7 影响的 R100 路由器 "
    f"[{answer_citations[0]}][{answer_citations[1]}]。"
    "图中只证明关系链，不证明客户已经遭受实际损失。"
)
print("KG-RAG context:\n" + kg_context)
print("\n受约束回答：", answer)
print("引用映射：", {key: value["source_uri"] for key, value in citation_map.items()})


## 8. 时间查询、更新与删除语义

“当前供应商”和“2024 年的供应商”是不同查询。事实边要保存有效区间，查询显式传 `as_of`。同一自然事实收到更晚的 `observed_at` 时，本例允许更正 `valid_to` 并记录 old/new 审计；旧消息重放不能覆盖新更正。完整双时间查询仍需同时支持“有效于何时”和“系统在何时已知”。

删除分为来源撤回、事实失效、GDPR/合规物理删除等不同语义。逻辑 tombstone 适合审计，但不一定满足必须物理清除的法规要求；生产方案需定义级联、索引清理、备份保留和引用失效策略。


In [ ]:
template_fact = next(f for f in extracted_facts if f.tenant == "tenant-a" and f.predicate == "SUPPLIES")
temporal_correction = replace(
    template_fact,
    source_id="supply-close-d7", source_uri="corrections/supply-close",
    observed_at="2026-08-01", valid_to="2026-07-31", confidence=0.995,
)
store.upsert_fact(temporal_correction)
active_before_close = {key for _, _, key, _ in store.active_facts("tenant-a", "2026-07-31")}
active_after_close = {key for _, _, key, _ in store.active_facts("tenant-a", "2026-08-01")}
temporal_update_audit = next(
    row for row in reversed(store.audit_log)
    if row["fact_id"] == template_fact.fact_id and row["operation"] == "update_temporal"
)

historical_fact = replace(
    template_fact,
    fact_id=stable_id("tenant-a", template_fact.subject_id, template_fact.predicate, template_fact.object_id, "2024-01-01"),
    source_id="history-d0", source_uri="archive/supply-2024", observed_at="2024-01-02",
    valid_from="2024-01-01", valid_to="2025-12-31", confidence=0.95,
)
store.upsert_fact(historical_fact)
suppliers_2024 = secure_one_hop(store, AUTH_A_READER, "product:r100", "SUPPLIES", "in", as_of="2024-06-01")
assert [entity["entity_id"] for entity in suppliers_2024] == ["org:huaxing"]
first_delete = store.delete_fact("tenant-a", historical_fact.fact_id, "2026-07-27T12:00:00+08:00")
second_delete = store.delete_fact("tenant-a", historical_fact.fact_id, "2026-07-27T12:00:01+08:00")
historical_deleted_state = next(data["status"] for _, _, key, data in store.graph.edges(keys=True, data=True) if key == historical_fact.fact_id)
print({
    "current_supply_valid_to": temporal_update_audit["changes"]["valid_to"]["new"],
    "active_on_2026-07-31": template_fact.fact_id in active_before_close,
    "active_on_2026-08-01": template_fact.fact_id in active_after_close,
    "2024_suppliers": [e["name"] for e in suppliers_2024],
    "first_delete_changed": first_delete, "repeat_delete_changed": second_delete,
    "state": historical_deleted_state,
})


## 9. 评估不能只数节点和边

推荐分层评估：

1. mention detection 的 span precision/recall；
2. entity linking 的 canonical ID accuracy、候选召回和拒识质量；
3. relation extraction 的严格三元组 precision/recall/F1，并分关系报告；
4. 图约束通过率、重复率、provenance 覆盖率、新鲜度；
5. 业务查询的 answer accuracy、path recall、evidence precision/recall 和引用可达率；
6. KG-RAG 的答案正确性、忠实度、引用正确性、拒答与越权率。

下面的 1.0 来自手写规则和同源 gold，只证明代码没有偏离这组受控样例，绝不代表对自由文本的泛化能力。


In [ ]:
gold_tenant_a = {
    ("org:huaxing", "SUPPLIES", "product:r100"),
    ("org:farsea", "BUYS", "product:r100"),
    ("org:farsea", "PARTY_TO", "contract:c202601"),
    ("org:huaxing", "PARTY_TO", "contract:c202601"),
    ("contract:c202601", "COVERS", "product:r100"),
    ("incident:inc7", "AFFECTS", "product:r100"),
}
predicted_tenant_a = {(f.subject_id, f.predicate, f.object_id) for f in extracted_facts if f.tenant == "tenant-a"}
tp = len(gold_tenant_a & predicted_tenant_a)
precision = tp / len(predicted_tenant_a)
recall = tp / len(gold_tenant_a)
triple_f1 = 2 * precision * recall / (precision + recall)
provenance_coverage = sum(bool(data["provenance"]) for _, _, _, data in store.active_facts("tenant-a")) / len(store.active_facts("tenant-a"))
query_answer_correct = {entity["entity_id"] for entity in affected_customers} == {"org:farsea"}
print({
    "strict_triple_precision": precision, "strict_triple_recall": recall, "strict_triple_f1": triple_f1,
    "provenance_coverage": provenance_coverage, "controlled_query_correct": query_answer_correct,
})


## 10. 安全、权限与多租户隔离

权限必须在**候选生成/图遍历之前**执行，不能先查全图再让 LLM“不要泄露”。本例所有公开 one-hop、路径和 context API 都接收由认证网关构造的可信 `AuthContext(tenant, scopes)`；tenant 不能来自请求正文。节点、边、provenance 和字段投影使用同一上下文，读取 source URI 还必须拥有 `kg.provenance.read`。

还要防御：来源文档提示注入、恶意实体/关系污染、LLM 生成的无界 Cypher/SPARQL、超深遍历资源耗尽，以及删除后缓存仍返回旧证据。查询模板应白名单关系和深度，写操作与高成本查询要限流和审计。


In [ ]:
AUTH_B_READER = AuthContext("tenant-b", frozenset({"kg.read"}), "reader-b")
AUTH_A_FINANCE = AuthContext(
    "tenant-a", frozenset({"kg.read", "contract.finance.read"}), "finance-a",
)

tenant_b_suppliers = secure_one_hop(
    store, AUTH_B_READER, "product:r100-b", "SUPPLIES", "in", as_of="2026-07-10",
)
masked_contract = secure_one_hop(
    store, AUTH_A_READER, "org:farsea", "PARTY_TO", "out", as_of="2026-07-10",
)[0]
finance_contract = secure_one_hop(
    store, AUTH_A_FINANCE, "org:farsea", "PARTY_TO", "out", as_of="2026-07-10",
)[0]

cross_tenant_blocked = False
try:
    constrained_path(
        store, AUTH_A_READER, "incident:inc7", "org:huaxing-b", "2026-07-10",
        INCIDENT_CUSTOMER_PATTERN, max_hops=2,
    )
except nx.NetworkXNoPath:
    cross_tenant_blocked = True

provenance_blocked = False
try:
    assemble_kg_context(store, AUTH_A_READER, evidence_path, "2026-07-10")
except PermissionError:
    provenance_blocked = True

print({
    "tenant_b_supplier_ids": [e["entity_id"] for e in tenant_b_suppliers],
    "masked_contract": masked_contract, "finance_contract": finance_contract,
    "cross_tenant_blocked": cross_tenant_blocked,
    "provenance_without_scope_blocked": provenance_blocked,
})


## 11. NetworkX 与生产图数据库的取舍

NetworkX 很适合算法原型、单机测试和小图分析：Python API 直观，路径算法丰富。但它是进程内对象，没有持久事务、并发写、分布式执行、原生 ACL 或在线备份，本例也只是线性扫描边。

生产选型取决于访问模式：

- 属性图 + 高频在线遍历：评估 Neo4j/Cypher、Amazon Neptune、JanusGraph 等的事务、索引和运维；
- 标准语义互操作、推理和约束：RDF/SPARQL + SHACL/OWL 更合适，但建模与查询成本不同；
- 事实主要是分析型批处理：表/湖仓保存 canonical facts，按需物化图可能更简单；
- 混合检索：图存结构，全文/向量索引存召回表示；用稳定 entity/fact ID 对齐，并设计一致性与重建策略。

选型压测必须使用真实规模、度分布、路径长度、读写比和 tenant 过滤，不能只比较一条 demo Cypher 的延迟。


## 12. 可执行不变量与测试策略

单元测试覆盖 normalization、歧义拒绝、规则 span、关系签名和稳定 ID；集成测试覆盖重复摄取、来源聚合、时间查询、tombstone、tenant 隔离与证据引用；离线评估覆盖真实标注集；在线监控覆盖摄取延迟、约束失败率、未解析 mention、热点节点、查询超时、空证据率和越权拒绝。

下面把本 Notebook 的核心契约固化为断言。任何 schema 或抽取器升级都应在新旧版本上重放同一 golden corpus，并解释差异。


In [ ]:
assert invalid_error and "关系签名错误" in invalid_error
assert resolve_mention("华星", "tenant-a", "Organization") is None
assert resolve_mention("R-100路由器", "tenant-a", "Product") == "product:r100"
assert Counter(f.fact_id for f in extracted_facts)[supply_fact_id] == 2
assert initial_supply_provenance_count == 2
assert set(supply_edge["provenance"]) == {"d1", "d5", "supply-close-d7"}
assert [entity["entity_id"] for entity in suppliers] == ["org:huaxing"]
assert {entity["entity_id"] for entity in affected_customers} == {"org:farsea"}
assert [(step.predicate, step.direction) for step in evidence_path] == [("AFFECTS", "out"), ("BUYS", "in")]
assert evidence_path[0].from_node.endswith("incident:inc7") and evidence_path[-1].to_node.endswith("org:farsea")
assert [item["predicate"] for item in citation_map.values()] == ["AFFECTS", "BUYS"]
assert all(item["fact_id"] in citation and f"[{citation}]" in answer for citation, item in citation_map.items())
assert all(item["source_id"] in citation for citation, item in citation_map.items())
assert precision == recall == triple_f1 == provenance_coverage == 1.0
assert template_fact.fact_id in active_before_close and template_fact.fact_id not in active_after_close
assert supply_edge["valid_to"] == "2026-07-31" and temporal_update_audit["operation"] == "update_temporal"
assert first_delete is True and second_delete is False and historical_deleted_state == "tombstoned"
assert [entity["entity_id"] for entity in tenant_b_suppliers] == ["org:huaxing-b"]
assert cross_tenant_blocked and provenance_blocked
assert "contract_value" not in masked_contract["properties"]
assert finance_contract["properties"]["contract_value"] == 800000
for _, _, _, data in store.active_facts("tenant-a"):
    assert data["tenant"] == "tenant-a" and data["provenance"] and 0 <= data["confidence"] <= 1
print("实体解析、约束、幂等、时间、路径、引用和权限断言全部通过。")


## 13. 参考资料与教学边界

- W3C 官方标准：[RDF 1.1 Concepts and Abstract Syntax](https://www.w3.org/TR/rdf11-concepts/)、[SPARQL 1.1 Query Language](https://www.w3.org/TR/sparql11-query/)、[Shapes Constraint Language (SHACL)](https://www.w3.org/TR/shacl/)。
- NetworkX 官方文档：[MultiDiGraph](https://networkx.org/documentation/stable/reference/classes/multidigraph.html) 与 [Shortest Paths](https://networkx.org/documentation/stable/reference/algorithms/shortest_paths.html)。
- Hogan et al., [Knowledge Graphs](https://dl.acm.org/doi/10.1145/3447772), ACM Computing Surveys 2021：知识图谱表示、抽取、质量与应用的系统综述。
- Lewis et al., [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://papers.nips.cc/paper/2020/hash/6b493230205f780e1bc26945df7481e5-Abstract.html), NeurIPS 2020。
- Edge et al., [From Local to Global: A Graph RAG Approach to Query-Focused Summarization](https://arxiv.org/abs/2404.16130), 2024。
- Neo4j 官方文档：[Constraints](https://neo4j.com/docs/cypher-manual/current/constraints/) 与 [MERGE](https://neo4j.com/docs/cypher-manual/current/clauses/merge/)，可用于对照生产唯一约束和幂等写入语义。

> 教学边界：本例的 alias 表、结构化规则、内存时态模型和 NetworkX 线性扫描只适用于小型演示。生产系统仍需持久事务、并发控制、来源版本/撤回协议、真正的实体链接评测、认证网关签发的 AuthContext、来源级 ACL、灾备与可回滚的 schema/数据迁移。
